In [1]:
import os
import cv2
import torch
import numpy as np

from tqdm import tqdm
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

/home/gokul/ideaslab/vggt_for_camera_params/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cam1 = "calibration_images/winston/cam1"
cam2 = "calibration_images/winston/cam2"
cam3 = "calibration_images/winston/cam3"

output = "calibration_images/winston/camera_params"

In [ ]:
model = VGGT.from_pretrained("facebook/VGGT-1B").to(DEVICE)
model.eval()

VGGT(
  (aggregator): Aggregator(
    (patch_embed): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
        (norm): Identity()
      )
      (blocks): ModuleList(
        (0-23): 24 x NestedTensorBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): MemEffAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (a

In [4]:
cam1_files = sorted(os.listdir(cam1))
cam2_files = sorted(os.listdir(cam2))
cam3_files = sorted(os.listdir(cam3))

common_files = list(set(cam1_files) & set(cam2_files) & set(cam3_files))

In [5]:
img = cv2.imread("calibration_images/new/cam1/frame_0000.jpg")

H, W = img.shape[:2]

In [6]:
all_intrinsics = []
all_extrinsics = []

with torch.no_grad():
    for fname in tqdm(common_files):
        paths = [
            os.path.join(cam1, fname),
            os.path.join(cam2, fname),
            os.path.join(cam3, fname),
        ]

        images = load_and_preprocess_images(paths).to(DEVICE)

        outputs = model(images)

        pose_enc = outputs["pose_enc"]

        extrinsics, intrinsics = pose_encoding_to_extri_intri(pose_enc, image_size_hw=(H, W))

        all_intrinsics.append(intrinsics[0].cpu().numpy())
        all_extrinsics.append(extrinsics[0].cpu().numpy())

        break

  0%|          | 0/100 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 54.00 MiB. GPU 0 has a total capacity of 15.57 GiB of which 40.38 MiB is free. Process 3433867 has 2.24 GiB memory in use. Process 4067074 has 8.13 GiB memory in use. Including non-PyTorch memory, this process has 5.12 GiB memory in use. Of the allocated memory 4.80 GiB is allocated by PyTorch, and 80.01 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
all_extrinsics = np.array(all_extrinsics)
all_intrinsics = np.array(all_intrinsics)

np.save(os.path.join(output, "extrinsics.npy"), all_extrinsics)
np.save(os.path.join(output, "intrinsics.npy"), all_intrinsics)

print(f" Extrisnics shape: {all_extrinsics.shape} \n Intrinsics shape: {all_intrinsics.shape}")

 Extrisnics shape: (0,) 
 Intrinsics shape: (0,)


In [ ]:
images.shape

torch.Size([3, 3, 420, 518])